# Projects Testing Lab

In [1]:
! uv add pytest pytest-cov black isort flake8 mypy pre-commit

Resolved 184 packages in 3ms
Checked 181 packages in 6ms


In [2]:
! pytest --version 
! black --version

pytest 9.0.3
black, 26.3.1 (compiled: yes)
Python (CPython) 3.12.13


run `run_buggy.py`: 

In [4]:
! uv run ../run_buggy.py

ROC-AUC : 0.4707
Le script s'execute sans erreur... mais est-il correct ?


Question A:  
to detect these bugs without tests, we need to read the whole code source to figure them out, and we also need to run some expirements for better detection of data leakage and stratisfaction absence detection.   
 
this takes a lot of time for every new member, depending on the size of the source code and the lvl of documentation.  

Etape 6: Running Tests

1- Buggy Code (with data  leakage, wrong target, no stratisfaction,...)

In [ ]:

# switch to repo root so pytest can find tests/
%cd ..
# run tests
!pytest -q
# with coverage
!pytest --cov=src --cov-report=term-missing -q

/home/zeco/Desktop
/home/zeco/Desktop
....F.F.F...FF..                                                         [100%]
=================================== FAILURES ===================================
____________ TestTrainTestSplit.test_stratified_split_class_balance ____________

self = <tests.test_data_loader.TestTrainTestSplit object at 0x7fab0d0b9070>
sample_df =      Pregnancies     Glucose  BloodPressure  ...  DiabetesPedigree  Age  Outcome
0              0  188.115490     102....   62        0
199            9  153.522826      81.019140  ...          1.033832   44        1

[200 rows x 9 columns]

    def test_stratified_split_class_balance(self, sample_df):
        """Le ratio de classes doit être préservé entre train et test (stratify)."""
        from src.data_loader import preprocess_data
        df = sample_df.copy()
        df['Outcome'] = 0
        df.loc[:79, 'Outcome'] = 1
        # exactement 40%
        _, _, y_train, y_test, _ = preprocess_data(df)
        train_ratio

Question B:  
- no stratisfy: no exception because the argument is optional. impact: wrong test metrics -> deploying a bad model.  

- data leakage: no exception because it's only related to the order of splitting & normalization. impact: model sees the testing data -> good testing metrics, wrong model deployed.  

- wrong target: not a python error to produce an exception, it's just a choice. impact: strange behavior, signature different from the one expected in prod.  

- using binary in ROC-AUC instead of probs: no exceptions because binary values are still numbers, impact: low ROC-AUC for a model that might be the best.  

- pipeline incomplet: pipeline steps are optional, no step is forced. impact: prod data not passing from the same preprocessing of the training data.  

*the most dangerous:* data leakage, because it gives good testing metrics, the model will be deployed and show bad performance. 

Clean code (no data leakage or bad metrics, etc.)

In [7]:


# run tests
!pytest -v
# with coverage
!pytest --cov=src --cov-report=term-missing -q

/home/zeco/Desktop/TPs/mlops/diabletes_mlops
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0 -- /home/zeco/Desktop/TPs/mlops/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/zeco/Desktop/TPs/mlops/diabletes_mlops
configfile: pytest.ini
testpaths: tests
plugins: hydra-core-1.3.2, anyio-4.13.0, cov-7.1.0
collected 16 items                                                             

tests/test_data_loader.py::TestZeroReplacement::test_no_nan_after_preprocessing PASSED [  6%]
tests/test_data_loader.py::TestZeroReplacement::test_glucose_no_zeros PASSED [ 12%]
tests/test_data_loader.py::TestZeroReplacement::test_output_types_are_numeric PASSED [ 18%]
tests/test_data_loader.py::TestTrainTestSplit::test_split_sizes PASSED   [ 25%]
tests/test_data_loader.py::TestTrainTestSplit::test_stratified_split_class_balance PASSED [ 31%]
tests/test_data_loader.py::TestTrainTestSplit::test_no_overlap_bet